# Silver Layer — Patient ICD Procedures (`procedures_icd`)

**Source -> Target:** `<source_schema>.procedures_icd` -> `<target_schema>.procedures_icd` (defaults `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`). The target is replaced via `CREATE OR REPLACE`; downstream Gold depends on it.

**Objective:** Standardize patient-level ICD procedure records while preserving patient identifiers, admission relationships, procedure dates, sequence numbers, and ICD-9/ICD-10 distinctions.

**Expected grain:** One recorded procedure assignment per patient, admission and procedure sequence. Per MIMIC-IV `hosp.procedures_icd`, `subject_id`, `hadm_id` and `seq_num` are NOT NULL; `icd_code`/`icd_version` link to `d_icd_procedures`. Uniqueness is asserted on the full six-column record (including `chartdate`, `icd_code`, `icd_version`); uniqueness of (`subject_id`, `hadm_id`, `seq_num`) is not separately asserted. No join to the dictionary is performed here.

**Source-file evidence (local MIMIC-IV CSV, 859,655 rows):** the documented 3-column key `(subject_id, hadm_id, seq_num)` is **not unique** in the data (859,650 distinct keys; 5 rows share a key), whereas the full 6-column record is unique (859,655 distinct) and no column contains NULLs. The unconditional uniqueness check is therefore on the 6-column record. See `docs/gold_governance_checks.md`, Source readiness.

**Silver rules:** Preserve all source records and clinical meaning, standardize data types, validate completeness and duplicate assignments, and reconcile Bronze–Silver counts. No imputation, automatic deduplication or analytical aggregation.

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
# Configure Bronze source and Silver target schemas (widgets allow standalone and dbutils.notebook.run use)
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TABLE = "procedures_icd"
SOURCE_TABLE = f"{BRONZE}.{TABLE}"
TARGET_TABLE = f"{SILVER}.{TABLE}"

# Expected published contract (ordered column -> type)
EXPECTED_SCHEMA = {
    "subject_id": "bigint",
    "hadm_id": "bigint",
    "seq_num": "int",
    "chartdate": "date",
    "icd_code": "string",
    "icd_version": "int",
}

In [0]:
# Load and inspect patient ICD procedure records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

print("Bronze rows:", bronze_df.count())

In [0]:
# Profile source row counts and missing values
bronze_profile = bronze_df.select(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in [
            "subject_id", "hadm_id", "seq_num",
            "chartdate", "icd_code", "icd_version"
        ]
    ]
)

display(bronze_profile)

In [0]:
# Standardize identifiers, procedure dates and ICD coding fields
silver_df = bronze_df.select(
    F.col("subject_id").cast("long").alias("subject_id"),
    F.col("hadm_id").cast("long").alias("hadm_id"),
    F.col("seq_num").cast("int").alias("seq_num"),
    F.col("chartdate").cast("date").alias("chartdate"),
    F.trim(F.col("icd_code").cast("string")).alias("icd_code"),
    F.col("icd_version").cast("int").alias("icd_version")
)

display(silver_df.limit(20))
silver_df.printSchema()

In [0]:
# Check exact duplicate procedure records without removing any rows
duplicate_df = (
    silver_df
    .groupBy(
        "subject_id", "hadm_id", "seq_num",
        "chartdate", "icd_code", "icd_version"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_df.count()

print("Exact duplicate procedure groups:", duplicate_count)
display(duplicate_df)

In [0]:
# Check missing identifiers, procedure dates and invalid ICD versions
quality_df = silver_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in [
            "subject_id", "hadm_id", "seq_num",
            "chartdate", "icd_code", "icd_version"
        ]
    ],
    F.sum(
        F.when(F.col("icd_code") == "", 1).otherwise(0)
    ).alias("blank_icd_codes"),
    F.sum(
        F.when(
            F.col("icd_version").isNotNull() &
            (~F.col("icd_version").isin(9, 10)), 1
        ).otherwise(0)
    ).alias("invalid_icd_versions")
)

display(quality_df)

In [0]:
# Confirm the transformation preserves every Bronze record
bronze_count = bronze_df.count()
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"

In [0]:
# Stop publication if critical procedure quality checks fail
quality = quality_df.first().asDict()

assert duplicate_count == 0, f"Duplicate procedure groups: {duplicate_count}"
assert all(value == 0 for value in quality.values()), f"Quality issues: {quality}"
assert bronze_count == silver_count, "Record counts do not match"

print("All critical Silver quality checks passed.")

In [0]:
# Publish the validated Silver procedure table (CREATE OR REPLACE; downstream Gold depends on it)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

In [0]:
# Verify published records against the validated Silver DataFrame
published_df = spark.table(TARGET_TABLE)

published_count = published_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Published rows:", published_count)

assert published_count == silver_count, "Published row-count mismatch"
published_schema = {f.name: f.dataType.simpleString() for f in published_df.schema.fields}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from contract: {published_schema}"
)

published_quality = published_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in ["subject_id", "hadm_id", "seq_num", "chartdate", "icd_code", "icd_version"]
    ]
).first()
assert all(v == 0 for v in published_quality.asDict().values()), f"Published NULLs: {published_quality}"

published_duplicates = (
    published_df
    .groupBy("subject_id", "hadm_id", "seq_num", "chartdate", "icd_code", "icd_version")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert published_duplicates == 0, f"Published duplicate procedure records: {published_duplicates}"

display(published_df.limit(20))

print("Silver procedures_icd table verified successfully.")

In [0]:
# Finish successfully only after every assertion above has passed
dbutils.notebook.exit(f"{TABLE}:OK")